# Topic 1: Big-O & Complexity Analysis

**Goal**: Learn to measure how fast (or slow) your code is — before you even run it.  
**Time**: ~3-4 hours  
**Prereqs**: Topic 0

---

## Why Does This Matter?

Imagine you have a list of 1 million items.
- A **good** algorithm processes it in under 1 second
- A **bad** algorithm takes over 11 **days**

Big-O tells you which one you're writing. It's the difference between code that works and code that's useful.

---

## What is Big-O?

Big-O describes how the **number of operations grows** as the input size `n` grows. It's not about exact time — it's about the *growth pattern*.

```
Think of it like speed limits:
  O(1)     = teleportation   — instant, regardless of distance
  O(log n) = express highway — doubles the distance, adds 1 minute
  O(n)     = walking         — double the distance, double the time
  O(n²)    = asking everyone — double the people, quadruple the conversations
```

---

## The Big-O Family (Fastest → Slowest)

```
┌────────────┬───────────────────┬──────────────────────────────────┐
│  Big-O     │  Name             │  Real-World Example              │
├────────────┼───────────────────┼──────────────────────────────────┤
│  O(1)      │  Constant         │  Looking up arr[5]               │
│  O(log n)  │  Logarithmic      │  Binary search in sorted list    │
│  O(n)      │  Linear           │  Scanning every item once        │
│  O(n log n)│  Linearithmic     │  Sorting (merge sort, timsort)   │
│  O(n²)     │  Quadratic        │  Comparing every pair            │
│  O(2ⁿ)     │  Exponential      │  All subsets of a set            │
│  O(n!)     │  Factorial        │  All permutations                │
└────────────┴───────────────────┴──────────────────────────────────┘
```

**The gap is staggering.** For n = 1,000,000:
- O(log n) = ~20 operations
- O(n) = 1,000,000 operations
- O(n²) = 1,000,000,000,000 operations (!!)

In [ ]:
import time

def measure_time(func, *args, runs=5):
    """Run func multiple times and return average elapsed time in ms."""
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        result = func(*args)
        end = time.perf_counter()
        times.append((end - start) * 1000)
    avg = sum(times) / len(times)
    return avg, result

print("measure_time ready!")
print("Usage: avg_ms, result = measure_time(my_func, arg1, arg2)")

---

## O(1) — Constant Time

No matter the input size, it takes the **same number of steps**.  
Like looking up a word in a phonebook when you know the exact page number.

```
Input size:    10      100      1,000      1,000,000
Operations:     1        1          1              1
                ▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
                          flat line forever
```

**Key insight**: The operation doesn't even *look* at most of the data.

In [ ]:
def get_first(lst):
    return lst[0]

def get_last(lst):
    return lst[-1]

def dict_lookup(d, key):
    return d[key]

# Test with wildly different sizes
sizes = [1_000, 100_000, 1_000_000, 10_000_000]

print("O(1) — List Indexing")
print("-" * 45)
for size in sizes:
    data = list(range(size))
    avg_ms, _ = measure_time(get_first, data)
    print(f"  n = {size:>10,}  →  {avg_ms:.4f} ms")

print()
print("O(1) — Dict Lookup")
print("-" * 45)
for size in sizes:
    data = {i: i*2 for i in range(size)}
    avg_ms, _ = measure_time(dict_lookup, data, size // 2)
    print(f"  n = {size:>10,}  →  {avg_ms:.4f} ms")

print()
print("Notice: time stays essentially the same regardless of size!")

---

## O(n) — Linear Time

You look at **every element once**. Double the input, double the time.  
Like reading a book page by page — no shortcuts.

```
n = 4:     ■ ■ ■ ■             →  4 operations
n = 8:     ■ ■ ■ ■ ■ ■ ■ ■     →  8 operations
n = 16:    ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■  →  16 operations
           ╚══════════════════════════════════╝
            grows proportionally with input
```

**Key insight**: You must touch each element at least once. Can't do better for unsorted data.

In [ ]:
def find_max(lst):
    maximum = lst[0]
    for item in lst:
        if item > maximum:
            maximum = item
    return maximum

def sum_all(lst):
    total = 0
    for item in lst:
        total += item
    return total

sizes = [10_000, 50_000, 100_000, 500_000, 1_000_000]

print("O(n) — find_max")
print("-" * 50)
for size in sizes:
    data = list(range(size))
    avg_ms, result = measure_time(find_max, data)
    print(f"  n = {size:>10,}  →  {avg_ms:8.3f} ms  (max = {result})")

print()
print("O(n) — sum_all")
print("-" * 50)
for size in sizes:
    data = list(range(size))
    avg_ms, result = measure_time(sum_all, data)
    print(f"  n = {size:>10,}  →  {avg_ms:8.3f} ms")

print()
print("Notice: doubling n roughly doubles the time!")

---

## O(n²) — Quadratic Time

**Nested loops**: for every element, you look at every other element.  
Like a round-robin tournament — everyone plays everyone.

```
n=4: ■ ■ ■ ■
     16 comparisons

n=8: ■ ■ ■ ■ ■ ■ ■ ■
     64 comparisons  (4x more, not 2x!)
```

**Why it explodes:**
```
n = 1,000    →       1,000,000 operations  (manageable)
n = 10,000   →     100,000,000 operations  (slow...)
n = 100,000  →  10,000,000,000 operations  (good luck)
```

**Key insight**: Often the first thing to optimize in an interview. Look for ways to eliminate the inner loop (usually with a hash set).

In [ ]:
import random

def has_duplicates_quadratic(lst):
    """O(n²) — compare every pair."""
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j]:
                return True
    return False

def has_duplicates_linear(lst):
    """O(n) — use a set to track seen items."""
    seen = set()
    for item in lst:
        if item in seen:
            return True
        seen.add(item)
    return False

print("HEAD-TO-HEAD: O(n²) vs O(n) duplicate detection")
print("=" * 60)
print(f"{'Size':<10} {'O(n²) time':<15} {'O(n) time':<15} {'Speedup':<10}")
print("-" * 60)

for size in [1000, 3000, 5000, 10000]:
    data = random.sample(range(size * 10), size)

    avg_quad, _ = measure_time(has_duplicates_quadratic, data, runs=3)
    avg_lin, _ = measure_time(has_duplicates_linear, data, runs=3)

    speedup = avg_quad / avg_lin if avg_lin > 0 else float('inf')
    print(f"  {size:<8} {avg_quad:>8.2f} ms    {avg_lin:>8.4f} ms    {speedup:>6.0f}x")

print()
print("The O(n) version doesn't just win — it DOMINATES.")

---

## O(log n) — Logarithmic Time

**Halves the problem each step.** The dictionary/phonebook analogy:  
you open to the middle, check if you need to go left or right, repeat.

```
Searching 1,000,000 items:
  Linear:  1,000,000 steps
  Binary:  20 steps (!)

Each step eliminates HALF the remaining items:
  1,000,000 → 500,000 → 250,000 → ... → 1   (only ~20 halvings)
```

**Why 20?** Because 2²⁰ = 1,048,576 ≈ 1 million.  
log₂(1,000,000) ≈ 20.

```
Step-by-step trace (searching for 7 in sorted list):

  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
                 ↑ mid=5, too low → go right
  [6, 7, 8, 9, 10]
        ↑ mid=8, too high → go left
  [6, 7]
     ↑ mid=6, too low → go right
  [7]
   ↑ FOUND! (4 steps instead of 7)
```

In [ ]:
def linear_search(lst, target):
    steps = 0
    for item in lst:
        steps += 1
        if item == target:
            return steps
    return steps

def binary_search(lst, target):
    steps = 0
    lo, hi = 0, len(lst) - 1
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if lst[mid] == target:
            return steps
        elif lst[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return steps

print("LINEAR SEARCH vs BINARY SEARCH (step counts)")
print("=" * 60)
print(f"{'n':<15} {'Linear steps':<18} {'Binary steps':<18} {'Ratio':<10}")
print("-" * 60)

for size in [100, 10_000, 1_000_000, 100_000_000]:
    target = size - 1
    data = list(range(size))

    lin_steps = linear_search(data, target)
    bin_steps = binary_search(data, target)

    ratio = lin_steps / bin_steps
    print(f"  {size:<13,} {lin_steps:<18,} {bin_steps:<18} {ratio:,.0f}x")

print()
print("Binary search is absurdly efficient on sorted data.")

---

## O(n log n) — The Sweet Spot

This is where **good sorting algorithms** live.  
It's like doing a log(n) operation for each of n items.

Python's built-in `sorted()` and `.sort()` use **Timsort** — an O(n log n) algorithm.

```
Why can't we sort faster than O(n log n)?
  → Proven mathematical lower bound for comparison-based sorts.
  → You MUST compare elements, and the minimum comparisons
    needed to fully order n items is n × log(n).

For n = 1,000,000:
  O(n)       =     1,000,000
  O(n log n) =    20,000,000  (only 20x more — very doable)
  O(n²)      = 1,000,000,000,000  (a trillion — not doable)
```

**Key insight**: If someone says "sort it first, then scan" — that's O(n log n) + O(n) = O(n log n) total.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = np.arange(1, 101)

complexities = {
    'O(1)':       np.ones_like(n, dtype=float),
    'O(log n)':   np.log2(n),
    'O(n)':       n.astype(float),
    'O(n log n)': n * np.log2(n),
    'O(n²)':      n.astype(float) ** 2,
}

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']

plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-darkgrid')

for (label, values), color in zip(complexities.items(), colors):
    plt.plot(n, values, label=label, linewidth=2.5, color=color)

plt.xlabel('Input Size (n)', fontsize=12)
plt.ylabel('Operations', fontsize=12)
plt.title('Big-O Growth Rates Compared', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='upper left')
plt.ylim(0, 500)
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

print("O(n²) explodes so fast it dwarfs everything else!")
print("In practice, O(n log n) and below are your targets.")

---

## The Growth Table

Numbers don't lie. Here's what happens as `n` grows:

In [ ]:
import math

def fmt(num):
    if num >= 1_000_000_000_000:
        return f"{num/1e12:.0f}T"
    elif num >= 1_000_000_000:
        return f"{num/1e9:.0f}B"
    elif num >= 1_000_000:
        return f"{num/1e6:.0f}M"
    elif num >= 1_000:
        return f"{num/1e3:.0f}K"
    else:
        return str(int(num))

sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]

print("THE GROWTH TABLE")
print("=" * 72)
header = f"{'n':<12} {'O(1)':<10} {'O(log n)':<10} {'O(n)':<10} {'O(n log n)':<12} {'O(n²)':<12}"
print(header)
print("-" * 72)

for n in sizes:
    o1 = 1
    olog = math.log2(n)
    on = n
    onlogn = n * math.log2(n)
    on2 = n * n

    print(f"  {fmt(n):<10} {fmt(o1):<10} {fmt(olog):<10} {fmt(on):<10} {fmt(onlogn):<12} {fmt(on2):<12}")

print()
print("At n=1M, O(n²) needs 1 TRILLION operations.")
print("At 1 billion ops/sec, that's ~17 MINUTES vs 0.02 seconds for O(n log n).")

---

## Space Complexity

Not just time — **memory** matters too.  
Space complexity = how much *extra* memory your algorithm uses.

```
┌────────────────┬─────────────────────────────────────────────┐
│  Space         │  What it means                              │
├────────────────┼─────────────────────────────────────────────┤
│  O(1) extra    │  Fixed number of variables (swap, counter)  │
│  O(n) extra    │  New list/dict proportional to input        │
│  O(n²) extra   │  2D matrix (adjacency matrix)              │
└────────────────┴─────────────────────────────────────────────┘
```

**Trade-off**: Often you can trade space for time.  
Using a hash set (O(n) space) can eliminate an inner loop (O(n²) → O(n) time).

In [ ]:
import sys

def reverse_new_list(lst):
    """O(n) space — creates a whole new list."""
    return lst[::-1]

def reverse_in_place(lst):
    """O(1) space — swaps elements using only index variables."""
    left, right = 0, len(lst) - 1
    while left < right:
        lst[left], lst[right] = lst[right], lst[left]
        left += 1
        right -= 1
    return lst

original = list(range(1_000_000))

print("SPACE COMPARISON: reverse_new_list vs reverse_in_place")
print("=" * 55)

copy1 = original[:]
result_new = reverse_new_list(copy1)
extra_space_new = sys.getsizeof(result_new)
print(f"  reverse_new_list:  extra space = {extra_space_new / 1024:.0f} KB  (new list)")

copy2 = original[:]
reverse_in_place(copy2)
print(f"  reverse_in_place:  extra space = ~16 bytes  (two int vars)")

print()
print("Both produce the same result, but in-place uses ~0 extra memory.")
print(f"  Verify: {copy2[:5]} ... {copy2[-5:]}")

---

## How to Analyze Your Code

### The 4 Rules

```
RULE 1: Drop constants
  O(2n) → O(n)        O(500) → O(1)

RULE 2: Keep only the dominant term
  O(n² + n) → O(n²)   O(n³ + n² + n) → O(n³)

RULE 3: Separate (sequential) loops → ADD
  for x in a:    ─┐
      ...         │ O(n)
  for x in b:    ─┤
      ...         │ O(m)
                  └→ Total: O(n + m)

RULE 4: Nested loops → MULTIPLY
  for x in a:        ─┐
      for y in b:     │ O(n × m)
          ...         │
                      └→ Total: O(n × m)
```

In [ ]:
# QUIZ: What's the Big-O of each function?
# Try to figure it out BEFORE reading the answer.

def mystery_1(lst):
    total = 0
    for item in lst:
        total += item
    for item in lst:
        total += item
    return total
# Answer: O(n) — two sequential loops, O(n + n) = O(2n) = O(n)

def mystery_2(lst):
    for i in range(len(lst)):
        for j in range(len(lst)):
            if lst[i] == lst[j] and i != j:
                return True
    return False
# Answer: O(n²) — nested loop, every pair compared

def mystery_3(n):
    count = 0
    while n > 1:
        n //= 2
        count += 1
    return count
# Answer: O(log n) — halving n each iteration

def mystery_4(lst):
    return lst[0] + lst[-1]
# Answer: O(1) — two index lookups, constant work

def mystery_5(lst):
    lst.sort()
    return lst[len(lst) // 2]
# Answer: O(n log n) — dominated by the sort

print("Run this cell, then check your answers above!")
print()
print("Results:")
print(f"  mystery_1([1,2,3]) = {mystery_1([1,2,3])}")
print(f"  mystery_2([1,2,3]) = {mystery_2([1,2,3])}")
print(f"  mystery_3(1024)    = {mystery_3(1024)} (log₂(1024) = 10)")
print(f"  mystery_4([1,2,3]) = {mystery_4([1,2,3])}")
print(f"  mystery_5([3,1,2]) = {mystery_5([3,1,2])}")

---

## Practice: Identify the Big-O

For each function below, determine:
1. **Time complexity**
2. **Space complexity**

Try before looking at the solutions!

In [ ]:
# Exercise 1
def exercise_1(lst):
    result = []
    for item in lst:
        if item not in result:
            result.append(item)
    return result

print("Exercise 1:", exercise_1([1, 2, 2, 3, 3, 3, 4]))
print("What's the time complexity? What's the space complexity?")

In [ ]:
# Exercise 2
def exercise_2(n):
    matrix = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(i * j)
        matrix.append(row)
    return matrix

print("Exercise 2 (n=4):")
for row in exercise_2(4):
    print(" ", row)
print("What's the time complexity? What's the space complexity?")

In [ ]:
# Exercise 3
def exercise_3(lst, target):
    seen = {}
    for i, num in enumerate(lst):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

print("Exercise 3:", exercise_3([2, 7, 11, 15], 9))
print("What's the time complexity? What's the space complexity?")

---

### Solutions

**Exercise 1:**
- Time: **O(n²)** — `item not in result` scans the result list (up to n items) for each of n items
- Space: **O(n)** — result list stores up to n unique items
- Better approach: use a set for O(1) lookups → total O(n) time

**Exercise 2:**
- Time: **O(n²)** — nested loop, n × n iterations
- Space: **O(n²)** — builds an n × n matrix
- This is inherently quadratic — you genuinely need n² cells

**Exercise 3:**
- Time: **O(n)** — single pass through the list, dict lookup is O(1)
- Space: **O(n)** — dict stores up to n entries
- This is the classic "Two Sum" problem — the hash map eliminates the need for a nested loop

---

## Summary Cheat Sheet

```
COMMON BIG-O IN PYTHON:

O(1):       dict/set lookup, list index, append/pop
O(log n):   binary search, balanced BST
O(n):       single loop, linear scan
O(n log n): sorting (sorted(), .sort())
O(n²):      nested loops, brute force pairs

INTERVIEW RULE OF THUMB:
  n ≤ 20        → O(2ⁿ) or O(n!) OK
  n ≤ 3,000     → O(n²) OK
  n ≤ 1,000,000 → need O(n) or O(n log n)
  n > 1,000,000 → need O(n) or O(log n)
```

---

**Next up**: Topic 2 — Arrays & Strings